# Third

In [18]:
"""
ad_final_analysis_full.py
AD 小鼠 Multiome 最终下游分析 (整合版) — 疾病特异 GRN 重布线

输出:
  1) 主图 ad_rewiring_main.png/pdf  —— 仅含可信面板:
       A_umap(伪时序UMAP) / Amat(相似度矩阵) / B / C / J(边数) /
       D / K(靶标分布) / E / L(动态密度) / I(汇总表)
  2) 各子图单独文件 (panels/panel_*.png/pdf)
  3) F/G/H 仍生成单图, 但带"区分度不足"标注, 归附录, 不进主图

依据诊断 (已验证):
  - 四模型 AUC > 0.99, 网络可信 (细胞类型差异 > 基因型差异)
  - 疾病信号 = 调控靶标身份重布线 (Jaccard~0.3)
  - 能力边界: TF调控广度不变 (r≈0.997); 边权重饱和不可用 -> 全部分析基于"边的身份"
  - TF级重布线分数区分度极低 (0.014~0.046) -> F/G/H 不可作单TF驱动者结论
"""
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ================================================================
# 配置
# ================================================================
BASE = "/home/wuyan/dygmamba_project/NewRealPlan/case2/data/AD/process/"
M = {
    "CRND8_MG": BASE + "model1_CRND8_Microglia/process/",
    "WT_MG":    BASE + "model2_WT_Microglia/process/",
    "CRND8_AS": BASE + "model3_CRND8_Astrocyte/process/",
    "WT_AS":    BASE + "model4_WT_Astrocyte/process/",
}
OUT = "/home/wuyan/dygmamba_project/DRIMA/data/case2/" + "figures/"
PANEL_DIR = OUT + "panels_third/"
os.makedirs(OUT, exist_ok=True)
os.makedirs(PANEL_DIR, exist_ok=True)

COL = {"CRND8": "#D6604D", "WT": "#4393C3",
       "MG": "#2166AC", "AS": "#762A83", "shared": "#999999"}
AD_TFS = ["Spi1", "Irf8", "Runx1", "Mef2c", "Cebpb", "Runx2",
          "Nfkb1", "Jun", "Fos", "Stat3", "Smad3", "Notch1", "Yap1"]
AUC = {"CRND8_MG": 0.994, "WT_MG": 0.995, "CRND8_AS": 0.994, "WT_AS": 0.996}
KEYS = ["CRND8_MG", "WT_MG", "CRND8_AS", "WT_AS"]
KLAB = ["CRND8\nMG", "WT\nMG", "CRND8\nAS", "WT\nAS"]

TOP_FRAC = 0.1
MIN_TARGETS = 30
REWIRING_SPREAD_MIN = 0.05   # 区分度阈值, 低于此 F/G/H 视为不可靠


# ================================================================
# 数据加载 + 复用计算
# ================================================================
def load():
    d = {}
    for k, p in M.items():
        sf, df_ = p + "tf_gene_v3.pkl", p + "tf_gene_dynamic_v3.pkl"
        d[k] = {
            "static": pd.read_pickle(sf) if os.path.exists(sf) else None,
            "dynamic": pickle.load(open(df_, "rb")) if os.path.exists(df_) else None,
            "adata_path": p + "rna_processed.h5ad",
        }
        if d[k]["static"] is None:
            print(f"  [警告] {k} 缺 tf_gene_v3.pkl")
    return d


def E(df):
    return set(zip(df["TF"], df["Gene"])) if df is not None else set()


def tf_rewiring_strong(d, c, w, top_frac=TOP_FRAC, min_targets=MIN_TARGETS):
    dc, dw = d[c]["static"], d[w]["static"]

    def top_per_tf(df):
        out = {}
        for tf, sub in df.groupby("TF"):
            if len(sub) < min_targets:
                continue
            k = max(min_targets, int(len(sub) * top_frac))
            out[tf] = set(sub.nlargest(k, "weight")["Gene"])
        return out

    tc, tw = top_per_tf(dc), top_per_tf(dw)
    res = {}
    for tf in set(tc) & set(tw):
        u = tc[tf] | tw[tf]
        if u:
            res[tf] = 1 - len(tc[tf] & tw[tf]) / len(u)
    return pd.Series(res).sort_values(ascending=False)


# ================================================================
# 通用辅助
# ================================================================
def lab(ax, t):
    ax.text(-0.12, 1.1, t, transform=ax.transAxes, fontsize=14, fontweight='bold')

def clean(ax):
    ax.spines[['top', 'right']].set_visible(False)


# ================================================================
# 面板: 伪时序 UMAP (新 A) —— 需各模型 rna_processed.h5ad
#   含 obsm['X_umap'] 和 obs['pseudotime']; 缺失则画占位提示
# ================================================================
def panel_A_umap(ax_or_fig, d, as_subgrid=None, with_label=True):
    """
    若 as_subgrid 提供 (fig, gridspec_cell), 则在该格内画 2x2 小UMAP;
    否则把传入的单个 ax 当作占位 (单图模式下另用 save_umap_single)。
    """
    try:
        import anndata as ad
    except Exception:
        ad = None
    files_ok = (ad is not None) and all(os.path.exists(d[k]["adata_path"]) for k in KEYS)

    if as_subgrid is not None:
        fig, cell = as_subgrid
        inner = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=cell,
                                                 hspace=0.35, wspace=0.3)
        title = {"CRND8_MG": "CRND8 MG", "WT_MG": "WT MG",
                 "CRND8_AS": "CRND8 Astro", "WT_AS": "WT Astro"}
        for idx, k in enumerate(KEYS):
            sub = fig.add_subplot(inner[idx // 2, idx % 2])
            if files_ok:
                a = ad.read_h5ad(d[k]["adata_path"])
                um = a.obsm["X_umap"]
                pt = a.obs["pseudotime"].values if "pseudotime" in a.obs else np.zeros(a.shape[0])
                sc = sub.scatter(um[:, 0], um[:, 1], c=pt, cmap="RdYlBu_r",
                                 s=2, alpha=0.6, rasterized=True)
                plt.colorbar(sc, ax=sub, shrink=0.6, pad=0.02)
            else:
                sub.text(0.5, 0.5, "UMAP\n(use existing\nPanel A PDF)",
                         ha='center', va='center', transform=sub.transAxes,
                         fontsize=7, color='gray')
            sub.set_title(title[k], fontsize=7,
                          color=COL["CRND8"] if "CRND8" in k else COL["WT"])
            sub.set_xticks([]); sub.set_yticks([])
        # 在子网格左上角放面板字母
        fig.text(cell.get_position(fig).x0 - 0.01,
                 cell.get_position(fig).y1 + 0.005, "A",
                 fontsize=14, fontweight='bold')
    return files_ok


def save_umap_single(d):
    """单独输出伪时序 UMAP 子图 (2x2)。"""
    try:
        import anndata as ad
    except Exception:
        print("  无 anndata, 跳过 UMAP 单图 (用现有 Panel A PDF)")
        return
    if not all(os.path.exists(d[k]["adata_path"]) for k in KEYS):
        print("  缺 rna_processed.h5ad, 跳过 UMAP 单图 (用现有 Panel A PDF)")
        return
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    title = {"CRND8_MG": "CRND8 MG", "WT_MG": "WT MG",
             "CRND8_AS": "CRND8 Astro", "WT_AS": "WT Astro"}
    for ax, k in zip(axes.ravel(), KEYS):
        a = ad.read_h5ad(d[k]["adata_path"])
        um = a.obsm["X_umap"]
        pt = a.obs["pseudotime"].values if "pseudotime" in a.obs else np.zeros(a.shape[0])
        sc = ax.scatter(um[:, 0], um[:, 1], c=pt, cmap="RdYlBu_r",
                        s=4, alpha=0.6, rasterized=True)
        plt.colorbar(sc, ax=ax, shrink=0.7, label="PT")
        ax.set_title(title[k], color=COL["CRND8"] if "CRND8" in k else COL["WT"])
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
    fig.tight_layout()
    fig.savefig(PANEL_DIR + "panel_A_umap.png", dpi=200, bbox_inches='tight')
    fig.savefig(PANEL_DIR + "panel_A_umap.pdf", bbox_inches='tight')
    plt.close(fig)
    print("  单图: panel_A_umap.png / .pdf")


# ================================================================
# 面板: 相似度矩阵 (原 A)
# ================================================================
def panel_simmat(ax, d, with_label=True, letter='B'):
    if with_label: lab(ax, letter)
    jm = np.zeros((4, 4))
    for i, a in enumerate(KEYS):
        for j, b in enumerate(KEYS):
            ea, eb = E(d[a]["static"]), E(d[b]["static"])
            jm[i, j] = len(ea & eb) / len(ea | eb) if (ea | eb) else 0
    im = ax.imshow(jm, cmap='YlOrRd', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, shrink=0.8, label='Jaccard')
    ax.set_xticks(range(4)); ax.set_xticklabels(KLAB, fontsize=8)
    ax.set_yticks(range(4)); ax.set_yticklabels(KLAB, fontsize=8)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{jm[i,j]:.2f}", ha='center', va='center',
                    fontsize=8, color='black' if jm[i, j] < 0.5 else 'white')
    ax.set_title("GRN similarity matrix\n(cell-type diff > genotype diff)",
                 fontsize=10, fontweight='bold')
    return jm


def panel_jaccard_bar(ax, jm, with_label=True, letter='C'):
    if with_label: lab(ax, letter)
    clean(ax)
    jmg, jas = jm[0, 1], jm[2, 3]
    ax.bar(["Microglia", "Astrocyte"], [jmg, jas],
           color=[COL["MG"], COL["AS"]], alpha=0.85)
    for i, v in enumerate([jmg, jas]):
        ax.text(i, v + 0.01, f"{v:.3f}", ha='center', fontsize=11)
    ax.set_ylabel("CRND8 vs WT Jaccard"); ax.set_ylim(0, 0.5)
    ax.set_title("Disease rewiring extent\n(lower = more rewired)",
                 fontsize=10, fontweight='bold')


def panel_edge_pct(ax, d, with_label=True, letter='D'):
    if with_label: lab(ax, letter)
    clean(ax)
    for pos, (ct, c, w) in enumerate([("Microglia", "CRND8_MG", "WT_MG"),
                                      ("Astrocyte", "CRND8_AS", "WT_AS")]):
        ec, ew = E(d[c]["static"]), E(d[w]["static"])
        sh, co, wo = len(ec & ew), len(ec - ew), len(ew - ec)
        tot = sh + co + wo
        ax.bar(pos, sh/tot*100, color=COL["shared"], label='Shared' if pos == 0 else None)
        ax.bar(pos, co/tot*100, bottom=sh/tot*100, color=COL["CRND8"],
               label='CRND8-specific' if pos == 0 else None)
        ax.bar(pos, wo/tot*100, bottom=(sh+co)/tot*100, color=COL["WT"],
               label='WT-specific' if pos == 0 else None)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Microglia", "Astrocyte"])
    ax.set_ylabel("% of edges"); ax.legend(fontsize=8)
    ax.set_title("GRN edge composition (%)\nCRND8 vs WT", fontsize=10, fontweight='bold')


def panel_edge_count(ax, d, with_label=True, letter='E'):
    """新 J: 绝对边数堆叠"""
    if with_label: lab(ax, letter)
    clean(ax)
    labels, shared, c_only, w_only = [], [], [], []
    for ct, c, w in [("Microglia", "CRND8_MG", "WT_MG"),
                     ("Astrocyte", "CRND8_AS", "WT_AS")]:
        ec, ew = E(d[c]["static"]), E(d[w]["static"])
        labels.append(ct); shared.append(len(ec & ew))
        c_only.append(len(ec - ew)); w_only.append(len(ew - ec))
    x = np.arange(len(labels))
    ax.bar(x, shared, 0.6, color=COL["shared"], label="Shared")
    ax.bar(x, c_only, 0.6, bottom=shared, color=COL["CRND8"], label="CRND8-specific")
    ax.bar(x, w_only, 0.6, bottom=np.array(shared)+np.array(c_only),
           color=COL["WT"], label="WT-specific")
    for i in range(len(labels)):
        ax.text(i, shared[i]/2, f"{shared[i]:,}", ha='center', va='center', fontsize=6, color='white')
        ax.text(i, shared[i]+c_only[i]/2, f"{c_only[i]:,}", ha='center', va='center', fontsize=6, color='white')
        ax.text(i, shared[i]+c_only[i]+w_only[i]/2, f"{w_only[i]:,}", ha='center', va='center', fontsize=6, color='white')
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("TF-Gene edge count"); ax.legend(fontsize=7)
    ax.set_title("Edge counts (absolute)\nshared vs genotype-specific",
                 fontsize=10, fontweight='bold')


def panel_breadth_scatter(ax, d, with_label=True, letter='F'):
    if with_label: lab(ax, letter)
    clean(ax)
    for ct, c, w, color in [("MG", "CRND8_MG", "WT_MG", COL["MG"]),
                            ("AS", "CRND8_AS", "WT_AS", COL["AS"])]:
        odc = d[c]["static"].groupby("TF")["Gene"].nunique()
        odw = d[w]["static"].groupby("TF")["Gene"].nunique()
        com = odc.index.intersection(odw.index)
        ax.scatter(odc[com], odw[com], s=8, alpha=0.4, color=color, label=ct)
    mx = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([0, mx], [0, mx], 'k--', lw=0.8, alpha=0.5)
    ax.set_xlabel("CRND8 TF target count"); ax.set_ylabel("WT TF target count")
    ax.legend(fontsize=8)
    ax.set_title("TF regulatory breadth conserved\n(r≈0.997: count unchanged)",
                 fontsize=10, fontweight='bold')


def panel_outdeg_hist(ax, d, with_label=True, letter='G'):
    """新 K: 靶标数分布"""
    if with_label: lab(ax, letter)
    clean(ax)
    for k, color, ls in [("CRND8_MG", COL["CRND8"], "-"), ("WT_MG", COL["WT"], "-"),
                         ("CRND8_AS", COL["CRND8"], "--"), ("WT_AS", COL["WT"], "--")]:
        s = d[k]["static"]
        if s is None:
            continue
        od = s.groupby("TF")["Gene"].nunique().values
        ax.hist(od, bins=40, histtype="step", lw=1.5, color=color, ls=ls,
                label=k.replace("_", " "), density=True)
    ax.set_xlabel("Targets per TF (out-degree)"); ax.set_ylabel("Density")
    ax.legend(fontsize=7)
    ax.set_title("TF regulatory breadth distribution", fontsize=10, fontweight='bold')


def panel_dyn_jaccard(ax, d, with_label=True, letter='H'):
    if with_label: lab(ax, letter)
    clean(ax)
    for ct, c, w, color in [("Microglia", "CRND8_MG", "WT_MG", COL["MG"]),
                            ("Astrocyte", "CRND8_AS", "WT_AS", COL["AS"])]:
        dc, dw = d[c]["dynamic"], d[w]["dynamic"]
        if not dc or not dw:
            continue
        nc, nw = max(dc.keys()), max(dw.keys())
        xs, ys = [], []
        for wc in sorted(dc.keys()):
            pos = wc / nc if nc else 0
            ww = min(dw.keys(), key=lambda x: abs(x/nw - pos) if nw else 0)
            ec = set(zip(dc[wc]["TF"], dc[wc]["Gene"]))
            ew = set(zip(dw[ww]["TF"], dw[ww]["Gene"]))
            u = ec | ew
            if u:
                xs.append(pos); ys.append(len(ec & ew)/len(u))
        ax.plot(xs, ys, 'o-', ms=4, color=color, label=ct)
    ax.set_xticks([0, 0.208, 0.695, 1.0])
    ax.set_xticklabels(["2.5m", "5.7m", "13.2m", "17.9m"], fontsize=8)
    ax.set_xlabel("Age (pseudotime)"); ax.set_ylabel("CRND8 vs WT Jaccard")
    ax.set_ylim(0, 0.6); ax.legend(fontsize=8)
    ax.set_title("GRN divergence over pseudotime", fontsize=10, fontweight='bold')


def panel_dyn_density(ax, d, with_label=True, letter='I'):
    """新 L: 每窗边数 (受细胞数影响)"""
    if with_label: lab(ax, letter)
    clean(ax)
    for k, color, ls in [("CRND8_MG", COL["CRND8"], "-"), ("WT_MG", COL["WT"], "-"),
                         ("CRND8_AS", COL["CRND8"], "--"), ("WT_AS", COL["WT"], "--")]:
        dyn = d[k]["dynamic"]
        if not dyn:
            continue
        wins = sorted(dyn.keys())
        counts = [len(dyn[w]) for w in wins]
        x = np.array(wins) / max(wins) if max(wins) > 0 else np.array(wins, dtype=float)
        ax.plot(x, counts, marker="o", ms=3, color=color, ls=ls, label=k.replace("_", " "))
    ax.set_xlabel("Pseudotime (normalized)"); ax.set_ylabel("Edges per window")
    ax.legend(fontsize=7)
    ax.set_title("Dynamic network density\n(window-based; affected by cell count)",
                 fontsize=10, fontweight='bold')


def panel_summary(ax, d, with_label=True, letter='J'):
    if with_label: lab(ax, letter)
    ax.axis('off')
    rows = []
    for k in KEYS:
        s = d[k]["static"]
        rows.append([k.replace("_", " "), f"{AUC[k]:.3f}",
                     str(s["TF"].nunique()), f"{len(s):,}"])
    tbl = ax.table(cellText=rows, colLabels=["Model", "AUC", "TFs", "Edges"],
                   loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.8)
    for ri, c in enumerate(['#FFDDC1', '#C1E1FF', '#FFDDC1', '#C1E1FF']):
        for ci in range(4):
            tbl[(ri+1, ci)].set_facecolor(c)
    ax.set_title("Model summary", fontsize=10, fontweight='bold', pad=20)
    ax.text(0.5, 0.05, "Finding: disease rewires target identity,\n"
            "not TF breadth (r≈0.997)",
            transform=ax.transAxes, ha='center', fontsize=8, style='italic', color='#555')


# ================================================================
# 面板: 单TF靶标身份差异 subnetwork (纯结构, 可信)
#   选一个靶标数适中、CRND8/WT 共有的 TF 作代表性示例;
#   靶标分三类着色: 共有 / CRND8特异 / WT特异; 位置随机(不表强弱)
# ================================================================
def pick_representative_tf(d, c, w, lo=80, hi=400):
    """挑一个 CRND8/WT 都有、且并集靶标数适中的 TF (便于可视化)。"""
    dc, dw = d[c]["static"], d[w]["static"]
    if dc is None or dw is None:
        return None
    tc = dc.groupby("TF")["Gene"].apply(set)
    tw = dw.groupby("TF")["Gene"].apply(set)
    common = tc.index.intersection(tw.index)
    cands = []
    for tf in common:
        u = len(tc[tf] | tw[tf])
        if lo <= u <= hi:
            cands.append((tf, u))
    if not cands:
        # 放宽: 取并集最接近 (lo+hi)/2 的
        mid = (lo + hi) / 2
        allu = [(tf, len(tc[tf] | tw[tf])) for tf in common]
        if not allu:
            return None
        return min(allu, key=lambda x: abs(x[1] - mid))[0]
    # 在适中范围内, 取并集最大的(信息多一点)
    return max(cands, key=lambda x: x[1])[0]


def panel_tf_subnetwork(ax, d, c, w, ct_label, tf=None, with_label=True, letter='K'):
    if with_label: lab(ax, letter)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    dc, dw = d[c]["static"], d[w]["static"]
    if dc is None or dw is None:
        ax.text(0.5, 0.5, "no data", ha='center', va='center', transform=ax.transAxes)
        return
    if tf is None:
        tf = pick_representative_tf(d, c, w)
    if tf is None:
        ax.text(0.5, 0.5, "no shared TF\nin target range", ha='center', va='center',
                transform=ax.transAxes, fontsize=9, color='gray')
        return

    tc = set(dc[dc["TF"] == tf]["Gene"])
    tw = set(dw[dw["TF"] == tf]["Gene"])
    shared = tc & tw
    c_only = tc - tw
    w_only = tw - tc

    rng = np.random.default_rng(0)  # 固定种子保证可复现
    def place(genes, r_lo, r_hi):
        n = len(genes)
        if n == 0:
            return np.empty((0, 2))
        theta = rng.uniform(0, 2 * np.pi, n)
        r = rng.uniform(r_lo, r_hi, n)
        return np.c_[r * np.cos(theta), r * np.sin(theta)]

    # 三类靶标放在不同半径环带, 仅为视觉分组, 不代表调控强弱
    p_sh = place(shared, 0.25, 0.55)
    p_c = place(c_only, 0.60, 0.95)
    p_w = place(w_only, 0.60, 0.95)

    # 边 (TF中心 -> 靶标), 浅灰
    for P in (p_sh, p_c, p_w):
        for x, y in P:
            ax.plot([0, x], [0, y], color="#dddddd", lw=0.3, zorder=1)
    # 靶标节点
    ax.scatter(p_sh[:, 0], p_sh[:, 1], s=14, c=COL["shared"], label=f"Shared ({len(shared)})", zorder=2)
    ax.scatter(p_c[:, 0], p_c[:, 1], s=14, c=COL["CRND8"], label=f"CRND8-specific ({len(c_only)})", zorder=2)
    ax.scatter(p_w[:, 0], p_w[:, 1], s=14, c=COL["WT"], label=f"WT-specific ({len(w_only)})", zorder=2)
    # 中心 TF
    ax.scatter([0], [0], s=260, c="#333333", zorder=3)
    ax.text(0, 0, tf, color="white", ha="center", va="center", fontsize=8, fontweight="bold", zorder=4)

    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")
    ax.legend(fontsize=7, loc="upper right", frameon=False)
    ax.set_title(f"{ct_label}: {tf} target identity\n(representative TF; position not strength)",
                 fontsize=10, fontweight="bold")


# ================================================================
# 面板 M: Top TF 的靶标身份组成 (分组堆叠条形图, 纯结构可信)
#   替代表格 tf_specific_target_counts.csv 的可视化:
#   横向比较若干 TF 在 CRND8/WT 间的 共有/CRND8特异/WT特异 靶标数
# ================================================================
def tf_target_identity(d, c, w, top_n=15):
    """返回按总靶标数排序的前 top_n 个共有 TF 的 共有/CRND8特异/WT特异 计数。"""
    dc, dw = d[c]["static"], d[w]["static"]
    if dc is None or dw is None:
        return pd.DataFrame()
    tc = dc.groupby("TF")["Gene"].apply(set)
    tw = dw.groupby("TF")["Gene"].apply(set)
    common = tc.index.intersection(tw.index)
    rows = []
    for tf in common:
        sc, sw = tc[tf], tw[tf]
        rows.append({"TF": tf, "shared": len(sc & sw),
                     "CRND8_specific": len(sc - sw), "WT_specific": len(sw - sc),
                     "total": len(sc | sw)})
    df = pd.DataFrame(rows).sort_values("total", ascending=False).head(top_n)
    return df.iloc[::-1]  # 反转, 让最大的在条形图顶部


def panel_tf_identity_bar(ax, d, c, w, ct_label, top_n=15, with_label=True, letter='M'):
    if with_label: lab(ax, letter)
    clean(ax)
    df = tf_target_identity(d, c, w, top_n)
    if df.empty:
        ax.text(0.5, 0.5, "no data", ha='center', va='center', transform=ax.transAxes)
        return
    y = np.arange(len(df))
    ax.barh(y, df["shared"], color=COL["shared"], label="Shared")
    ax.barh(y, df["CRND8_specific"], left=df["shared"], color=COL["CRND8"], label="CRND8-specific")
    ax.barh(y, df["WT_specific"], left=df["shared"] + df["CRND8_specific"],
            color=COL["WT"], label="WT-specific")
    ax.set_yticks(y); ax.set_yticklabels(df["TF"], fontsize=7)
    ax.set_xlabel("Target gene count")
    ax.legend(fontsize=7, loc="lower right")
    ax.set_title(f"{ct_label}: per-TF target identity\n(top {top_n} TFs by total targets)",
                 fontsize=10, fontweight='bold')


# ================================================================
# 面板 O: 弥漫性重塑 — 全 TF 靶标身份分化度分布 (无监督, 纯边身份)
#   identity_divergence(TF) = 1 - Jaccard(S_CRND8, S_WT), 用全部靶标
#   展示: 全员高分化、TF间区分度低 -> 无离群驱动者, 疾病弥漫重塑
#   (诊断结果: MG spread=0.111, AS spread=0.095, 均 < 0.15 区分度阈值)
# ================================================================
DIV_MIN_TARGETS = 20  # 两侧靶标都需 >= 此数才纳入

def identity_divergence(d, c, w, min_targets=DIV_MIN_TARGETS):
    """对 CRND8/WT 共有 TF 算 1-Jaccard(全部靶标身份)。返回降序 Series。"""
    dc, dw = d[c]["static"], d[w]["static"]
    if dc is None or dw is None:
        return pd.Series(dtype=float)
    tc = dc.groupby("TF")["Gene"].apply(set)
    tw = dw.groupby("TF")["Gene"].apply(set)
    res = {}
    for tf in set(tc.index) & set(tw.index):
        sc, sw = tc[tf], tw[tf]
        u = sc | sw
        if len(sc) >= min_targets and len(sw) >= min_targets and u:
            res[tf] = 1 - len(sc & sw) / len(u)
    return pd.Series(res).sort_values(ascending=False)


def panel_diffuse_rewiring(ax, d, with_label=True, letter='O'):
    """小提琴+抖动散点: MG/AS 全 TF 分化度分布, 标注区分度(分位差)。"""
    if with_label: lab(ax, letter)
    clean(ax)
    data, labels, colors = [], [], []
    for name, c, w, col in [("Microglia", "CRND8_MG", "WT_MG", COL["MG"]),
                            ("Astrocyte", "CRND8_AS", "WT_AS", COL["AS"])]:
        div = identity_divergence(d, c, w)
        if len(div) == 0:
            continue
        data.append(div.values); labels.append(name); colors.append(col)
    if not data:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        return
    parts = ax.violinplot(data, showmedians=True, showextrema=True, widths=0.7)
    for i, b in enumerate(parts["bodies"]):
        b.set_facecolor(colors[i]); b.set_alpha(0.35); b.set_edgecolor(colors[i])
    for key in ("cmedians", "cmaxes", "cmins", "cbars"):
        if key in parts:
            parts[key].set_color("#444"); parts[key].set_linewidth(1)
    rng = np.random.default_rng(0)
    for i, arr in enumerate(data):
        x = (i + 1) + rng.uniform(-0.18, 0.18, len(arr))
        ax.scatter(x, arr, s=4, color=colors[i], alpha=0.25, zorder=3, rasterized=True)
    for i, arr in enumerate(data):
        spread = np.quantile(arr, 0.9) - np.quantile(arr, 0.1)
        ax.text(i + 1, max(arr) + 0.02, f"n={len(arr)}\nspread={spread:.3f}",
                ha="center", va="bottom", fontsize=7, color=colors[i])
    ax.set_xticks(range(1, len(labels) + 1)); ax.set_xticklabels(labels)
    ax.set_ylabel("Target identity divergence\n(1 - Jaccard, per TF)")
    ax.set_ylim(0, 1)
    ax.axhspan(0.6, 0.9, color="#cccccc", alpha=0.15, zorder=0)
    ax.set_title("Diffuse rewiring across all TFs\n"
                 "(every TF highly rewired; no outlier driver)",
                 fontsize=10, fontweight="bold")


# ---- 附录: TF重布线 (不可靠, 带红色标注) ----
def panel_rewiring_supp(ax, rw, title, reliable, with_label=True, letter='F'):
    if with_label: lab(ax, letter)
    clean(ax)
    ad_in = [t for t in rw.index if t in AD_TFS]
    show = list(dict.fromkeys(ad_in[:8] + rw.head(15).index.tolist()))[:15][::-1]
    vals = [rw[t] for t in show]
    cols = [COL["CRND8"] if t in AD_TFS else '#bbb' for t in show]
    ax.barh(range(len(show)), vals, color=cols, alpha=0.85)
    ax.set_yticks(range(len(show)))
    ax.set_yticklabels([f"$\\bf{{{t}}}$" if t in AD_TFS else t for t in show], fontsize=8)
    ax.set_xlabel("Rewiring score (strong targets)"); ax.set_xlim(0, 1)
    ax.set_title(title, fontsize=10, fontweight='bold')
    if not reliable:
        ax.text(0.5, 0.5, "low discriminative power\n(all TFs ~equally rewired)",
                transform=ax.transAxes, ha='center', va='center',
                fontsize=10, color='red', alpha=0.5, fontweight='bold', rotation=15)


def panel_known_ad_supp(ax, rw_mg, rw_as, reliable, with_label=True, letter='H'):
    if with_label: lab(ax, letter)
    clean(ax)
    present = [t for t in AD_TFS if t in rw_mg.index or t in rw_as.index]
    x = np.arange(len(present))
    mg_v = [rw_mg.get(t, np.nan) for t in present]
    as_v = [rw_as.get(t, np.nan) for t in present]
    ax.bar(x - 0.2, mg_v, 0.38, color=COL["MG"], label="Microglia")
    ax.bar(x + 0.2, as_v, 0.38, color=COL["AS"], label="Astrocyte")
    ax.set_xticks(x); ax.set_xticklabels(present, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel("Rewiring score (strong targets)"); ax.set_ylim(0, 1); ax.legend(fontsize=8)
    ax.set_title("Known AD TF rewiring\n(CRND8 vs WT)", fontsize=10, fontweight='bold')
    if not reliable:
        ax.text(0.5, 0.5, "low discriminative power",
                transform=ax.transAxes, ha='center', va='center',
                fontsize=11, color='red', alpha=0.5, fontweight='bold', rotation=15)


def save_single(name, draw_fn, figsize=(6, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    draw_fn(ax)
    fig.tight_layout()
    fig.savefig(PANEL_DIR + f"panel_{name}.png", dpi=200, bbox_inches='tight')
    fig.savefig(PANEL_DIR + f"panel_{name}.pdf", bbox_inches='tight')
    plt.close(fig)
    print(f"  单图: panel_{name}.png")


# ================================================================
# 单图导出 + 主程序
# ================================================================
def _jm(d):
    jm = np.zeros((4, 4))
    for i, a in enumerate(KEYS):
        for j, b in enumerate(KEYS):
            ea, eb = E(d[a]["static"]), E(d[b]["static"])
            jm[i, j] = len(ea & eb) / len(ea | eb) if (ea | eb) else 0
    return jm


# 重新实现单图导出 (避免上面的绕弯), 独立于 main 的主图流程
def export_summary_table(d):
    """把原图J(模型汇总)整理成论文用表格: CSV + Markdown + 一张表格图(可选放正文)。"""
    rows = []
    for k in KEYS:
        s = d[k]["static"]
        rows.append({
            "Model": k.replace("_", " "),
            "AUC": f"{AUC[k]:.3f}",
            "TFs": s["TF"].nunique() if s is not None else "-",
            "Edges": f"{len(s):,}" if s is not None else "-",
        })
    df = pd.DataFrame(rows)
    table_dir = OUT + "tables/"
    os.makedirs(table_dir, exist_ok=True)
    df.to_csv(table_dir + "model_summary_table.csv", index=False)
    # Markdown 版 (直接贴进论文)
    with open(table_dir + "model_summary_table.md", "w") as f:
        f.write("| Model | AUC | TFs | Edges |\n|---|---|---|---|\n")
        for _, r in df.iterrows():
            f.write(f"| {r['Model']} | {r['AUC']} | {r['TFs']} | {r['Edges']} |\n")
    print(f"  汇总表 -> {table_dir}model_summary_table.csv / .md")
    return df


def make_main_figure(d):
    """论文主图版: 7 个核心面板 (无 J)。
       行1 A(UMAP) B(矩阵) / 行2 E(边数) F(散点) / 行3 I(动态密度) O(弥漫) /
       行4 K(MG子网, 跨两列)"""
    print("\n生成论文主图版 (7面板, K跨列)...")
    fig = plt.figure(figsize=(13, 22))
    gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.5, wspace=0.32,
                           left=0.07, right=0.95, top=0.94, bottom=0.04)
    # 行1
    panel_A_umap(None, d, as_subgrid=(fig, gs[0, 0]))
    panel_simmat(fig.add_subplot(gs[0, 1]), d, letter='B')
    # 行2
    panel_edge_count(fig.add_subplot(gs[1, 0]), d, letter='E')
    panel_breadth_scatter(fig.add_subplot(gs[1, 1]), d, letter='F')
    # 行3
    panel_dyn_density(fig.add_subplot(gs[2, 0]), d, letter='I')
    panel_diffuse_rewiring(fig.add_subplot(gs[2, 1]), d, letter='O')
    # 行4: K 跨两列居中
    panel_tf_subnetwork(fig.add_subplot(gs[3, :]), d, "CRND8_MG", "WT_MG",
                        "Microglia", letter='K')
    fig.suptitle("AD Mouse Multiome — Disease-specific GRN Rewiring",
                 fontsize=14, fontweight='bold', y=0.955)
    fig.savefig(OUT + "ad_main_figure.png", dpi=300, bbox_inches='tight')
    fig.savefig(OUT + "ad_main_figure.pdf", bbox_inches='tight')
    plt.close(fig)
    print(f"  论文主图: {OUT}ad_main_figure.png / .pdf")


def make_supp_figure(d, rw_mg, rw_as, rel_mg, rel_as):
    """补充图版: 主图未用的可信面板 + 附录(不可靠)面板。
       C D G / H L(AS子网) M / N supp1 supp2"""
    print("\n生成补充图版...")
    fig = plt.figure(figsize=(19, 18))
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.38,
                           left=0.06, right=0.96, top=0.93, bottom=0.05)
    jm = _jm(d)
    panel_jaccard_bar(fig.add_subplot(gs[0, 0]), jm, letter='C')
    panel_edge_pct(fig.add_subplot(gs[0, 1]), d, letter='D')
    panel_outdeg_hist(fig.add_subplot(gs[0, 2]), d, letter='G')
    panel_dyn_jaccard(fig.add_subplot(gs[1, 0]), d, letter='H')
    panel_tf_subnetwork(fig.add_subplot(gs[1, 1]), d, "CRND8_AS", "WT_AS",
                        "Astrocyte", letter='L')
    panel_tf_identity_bar(fig.add_subplot(gs[1, 2]), d, "CRND8_MG", "WT_MG",
                          "Microglia", letter='M')
    panel_tf_identity_bar(fig.add_subplot(gs[2, 0]), d, "CRND8_AS", "WT_AS",
                          "Astrocyte", letter='N')
    panel_rewiring_supp(fig.add_subplot(gs[2, 1]), rw_mg,
                        "Microglia: TF rewiring (red=AD TF)", rel_mg, letter='S1')
    panel_known_ad_supp(fig.add_subplot(gs[2, 2]), rw_mg, rw_as,
                        rel_mg and rel_as, letter='S2')
    fig.suptitle("Supplementary — supporting and method-limitation panels",
                 fontsize=14, fontweight='bold', y=0.955)
    fig.savefig(OUT + "ad_supp_figure.png", dpi=300, bbox_inches='tight')
    fig.savefig(OUT + "ad_supp_figure.pdf", bbox_inches='tight')
    plt.close(fig)
    print(f"  补充图: {OUT}ad_supp_figure.png / .pdf")
    # supp3 (AS rewiring) 单独出图
    save_single("supp_ASrewiring_S3",
                lambda ax: panel_rewiring_supp(ax, rw_as,
                    "Astrocyte: TF rewiring (red=AD TF)", rel_as, with_label=False),
                (6, 5.5))


def export_singles(d, rw_mg, rw_as, rel_mg, rel_as):
    print("\n生成单独子图...")
    save_umap_single(d)
    jm = _jm(d)
    save_single("Bmat", lambda ax: panel_simmat(ax, d, with_label=False), (6, 5.5))
    save_single("C_jaccard", lambda ax: panel_jaccard_bar(ax, jm, with_label=False))
    save_single("D_edgepct", lambda ax: panel_edge_pct(ax, d, with_label=False))
    save_single("E_edgecount", lambda ax: panel_edge_count(ax, d, with_label=False))
    save_single("F_breadth", lambda ax: panel_breadth_scatter(ax, d, with_label=False))
    save_single("G_outdeg", lambda ax: panel_outdeg_hist(ax, d, with_label=False), (6.5, 5))
    save_single("H_dynjac", lambda ax: panel_dyn_jaccard(ax, d, with_label=False))
    save_single("I_dyndens", lambda ax: panel_dyn_density(ax, d, with_label=False), (6.5, 5))
    save_single("J_summary", lambda ax: panel_summary(ax, d, with_label=False), (6, 4))
    # 单TF靶标身份 subnetwork (纯结构, 可信)
    save_single("K_MG_subnetwork",
                lambda ax: panel_tf_subnetwork(ax, d, "CRND8_MG", "WT_MG", "Microglia",
                                               with_label=False), (6, 6))
    save_single("L_AS_subnetwork",
                lambda ax: panel_tf_subnetwork(ax, d, "CRND8_AS", "WT_AS", "Astrocyte",
                                               with_label=False), (6, 6))
    # per-TF 靶标身份组成条形图 (替代表格的可视化, 纯结构可信)
    save_single("M_MG_tf_identity",
                lambda ax: panel_tf_identity_bar(ax, d, "CRND8_MG", "WT_MG", "Microglia",
                                                 with_label=False), (7, 6))
    save_single("N_AS_tf_identity",
                lambda ax: panel_tf_identity_bar(ax, d, "CRND8_AS", "WT_AS", "Astrocyte",
                                                 with_label=False), (7, 6))
    # 图O: 弥漫性重塑分布 (无监督全TF) + 导出盲推分化度表 + 区分度诊断
    save_single("O_diffuse_rewiring",
                lambda ax: panel_diffuse_rewiring(ax, d, with_label=False), (6, 5.5))
    table_dir = OUT + "tables/"
    os.makedirs(table_dir, exist_ok=True)
    for nm, c, w in [("MG", "CRND8_MG", "WT_MG"), ("AS", "CRND8_AS", "WT_AS")]:
        div = identity_divergence(d, c, w)
        if len(div) == 0:
            continue
        div.to_csv(table_dir + f"identity_divergence_{nm}.csv", header=["divergence"])
        sp = div.quantile(0.9) - div.quantile(0.1)
        ok = sp >= 0.15
        print(f"  [{nm}] 分化度: n={len(div)} min={div.min():.3f} "
              f"med={div.median():.3f} max={div.max():.3f} spread={sp:.3f} "
              f"-> {'可作发现' if ok else '区分度不足: 弥漫性重塑, 非单TF驱动'}")
    # 附录: 不可靠的 TF 重布线图
    save_single("supp_MGrewiring",
                lambda ax: panel_rewiring_supp(ax, rw_mg, "Microglia: TF rewiring (red=AD TF)",
                                               rel_mg, with_label=False), (6, 5.5))
    save_single("supp_ASrewiring",
                lambda ax: panel_rewiring_supp(ax, rw_as, "Astrocyte: TF rewiring (red=AD TF)",
                                               rel_as, with_label=False), (6, 5.5))
    save_single("supp_knownAD",
                lambda ax: panel_known_ad_supp(ax, rw_mg, rw_as, rel_mg and rel_as,
                                               with_label=False), (7, 5))


if __name__ == "__main__":
    print("加载网络...")
    d = load()
    for k in KEYS:
        s = d[k]["static"]
        print(f"  {k}: {'%d边' % len(s) if s is not None else 'None'}")

    rw_mg = tf_rewiring_strong(d, "CRND8_MG", "WT_MG")
    rw_as = tf_rewiring_strong(d, "CRND8_AS", "WT_AS")
    sp_mg = (rw_mg.quantile(0.9) - rw_mg.quantile(0.1)) if len(rw_mg) else 0
    sp_as = (rw_as.quantile(0.9) - rw_as.quantile(0.1)) if len(rw_as) else 0
    rel_mg = sp_mg >= REWIRING_SPREAD_MIN
    rel_as = sp_as >= REWIRING_SPREAD_MIN
    print(f"  MG重布线区分度={sp_mg:.3f} ({'可靠' if rel_mg else '不可靠 -> 附录'})")
    print(f"  AS重布线区分度={sp_as:.3f} ({'可靠' if rel_as else '不可靠 -> 附录'})")

    # ---------- 主图: 5x3, 仅可信面板 ----------
    print("\n生成主图 (可信面板)...")
    fig = plt.figure(figsize=(20, 25))
    gs = gridspec.GridSpec(5, 3, figure=fig, hspace=0.6, wspace=0.38,
                           left=0.06, right=0.96, top=0.94, bottom=0.04)
    panel_A_umap(None, d, as_subgrid=(fig, gs[0, 0]))
    jm = panel_simmat(fig.add_subplot(gs[0, 1]), d, letter='B')
    panel_jaccard_bar(fig.add_subplot(gs[0, 2]), jm, letter='C')
    panel_edge_pct(fig.add_subplot(gs[1, 0]), d, letter='D')
    panel_edge_count(fig.add_subplot(gs[1, 1]), d, letter='E')
    panel_breadth_scatter(fig.add_subplot(gs[1, 2]), d, letter='F')
    panel_outdeg_hist(fig.add_subplot(gs[2, 0]), d, letter='G')
    panel_dyn_jaccard(fig.add_subplot(gs[2, 1]), d, letter='H')
    panel_dyn_density(fig.add_subplot(gs[2, 2]), d, letter='I')
    # 第4行: K(MG靶标身份subnetwork) / J(汇总表) / L(AS靶标身份subnetwork)
    panel_tf_subnetwork(fig.add_subplot(gs[3, 0]), d, "CRND8_MG", "WT_MG",
                        "Microglia", letter='K')
    panel_summary(fig.add_subplot(gs[3, 1]), d, letter='J')
    panel_tf_subnetwork(fig.add_subplot(gs[3, 2]), d, "CRND8_AS", "WT_AS",
                        "Astrocyte", letter='L')
    # 第5行: M(MG per-TF靶标身份) / O(弥漫性重塑分布) / N(AS per-TF靶标身份)
    panel_tf_identity_bar(fig.add_subplot(gs[4, 0]), d, "CRND8_MG", "WT_MG",
                          "Microglia", letter='M')
    panel_diffuse_rewiring(fig.add_subplot(gs[4, 1]), d, letter='O')
    panel_tf_identity_bar(fig.add_subplot(gs[4, 2]), d, "CRND8_AS", "WT_AS",
                          "Astrocyte", letter='N')
    fig.suptitle("AD Mouse Multiome — Disease-specific GRN Rewiring\n"
                 "Genotype rewires regulatory target identity (Jaccard~0.3), "
                 "TF breadth conserved (r≈0.997)",
                 fontsize=14, fontweight='bold', y=0.96)
    fig.savefig(OUT + "ad_rewiring_main.png", dpi=200, bbox_inches='tight')
    fig.savefig(OUT + "ad_rewiring_main.pdf", bbox_inches='tight')
    plt.close(fig)
    print(f"  主图: {OUT}ad_rewiring_main.png / .pdf")

    export_singles(d, rw_mg, rw_as, rel_mg, rel_as)

    # ---------- 论文用: 主图版(7面板,无J) + 补充图版 + 汇总表 ----------
    export_summary_table(d)
    make_main_figure(d)
    make_supp_figure(d, rw_mg, rw_as, rel_mg, rel_as)

    print("\n完成。")
    print(f"  主图 (可信面板) -> {OUT}ad_rewiring_main.png / .pdf")
    print(f"  可信子图 -> {PANEL_DIR}panel_A_umap, Bmat, C_jaccard, D_edgepct,")
    print(f"             E_edgecount, F_breadth, G_outdeg, H_dynjac, I_dyndens, J_summary")
    print(f"  附录(不可靠) -> {PANEL_DIR}panel_supp_MGrewiring / supp_ASrewiring / supp_knownAD")
    print("\n注意: supp_* 三图区分度不足, 仅作方法局限展示, 不应作为单TF驱动者结论。")

加载网络...
  CRND8_MG: 905214边
  WT_MG: 905908边
  CRND8_AS: 912599边
  WT_AS: 934188边
  MG重布线区分度=0.014 (不可靠 -> 附录)
  AS重布线区分度=0.046 (不可靠 -> 附录)

生成主图 (可信面板)...
  主图: /home/wuyan/dygmamba_project/DRIMA/data/case2/figures/ad_rewiring_main.png / .pdf

生成单独子图...
  单图: panel_A_umap.png / .pdf
  单图: panel_Bmat.png
  单图: panel_C_jaccard.png
  单图: panel_D_edgepct.png
  单图: panel_E_edgecount.png
  单图: panel_F_breadth.png
  单图: panel_G_outdeg.png
  单图: panel_H_dynjac.png
  单图: panel_I_dyndens.png
  单图: panel_J_summary.png
  单图: panel_K_MG_subnetwork.png
  单图: panel_L_AS_subnetwork.png
  单图: panel_M_MG_tf_identity.png
  单图: panel_N_AS_tf_identity.png
  单图: panel_O_diffuse_rewiring.png
  [MG] 分化度: n=848 min=0.631 med=0.685 max=0.819 spread=0.111 -> 区分度不足: 弥漫性重塑, 非单TF驱动
  [AS] 分化度: n=847 min=0.675 med=0.743 max=0.871 spread=0.095 -> 区分度不足: 弥漫性重塑, 非单TF驱动
  单图: panel_supp_MGrewiring.png
  单图: panel_supp_ASrewiring.png
  单图: panel_supp_knownAD.png
  汇总表 -> /home/wuyan/dygmamba_project/DRIMA/data/case2/fig

# function analysis

## Data generate

In [22]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
DRIMA Case-2 (AD) — focused re-tuning of disease-specific enrichment
====================================================================
Tries the two STATISTICALLY-LEGITIMATE fixes for the weak/off-target AD signal:

  Fix 1  BACKGROUND = genes expressed in THIS cell type (microglia / astrocyte),
         not genome-wide. This suppresses housekeeping/ribosomal contamination
         (the 'cytoplasmic translation' artefact in microglia) so disease-
         specific programs can surface.
  Fix 2  FOCUSED LIBRARIES tilted toward neuro-immune / disease biology
         (Reactome, GO-BP, KEGG, Hallmark) so weak signal is not diluted.

Honest stance: this is a legitimate background correction, NOT p-hacking.
If after this the CRND8-specific poles are still off-target (microglia ->
ribosome/translation; astrocyte -> scattered), the correct reading is that AD
regulatory remodelling is diffuse/combinatorial, and the manuscript should say
so rather than force a pathway story. Report nominal-p 'trends' ONLY with an
explicit 'not FDR-significant / hypothesis-generating' label.

Run after drima_enrichment_v2.py (reuses its delta-specific / delta-rank logic).

Deps: numpy pandas matplotlib gseapy>=1.0 anndata
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SIG = 0.05

# focused libraries (disease / neuro-immune tilted)
LIBS_FOCUS_MOUSE = ["Reactome_2022", "GO_Biological_Process_2021",
                    "KEGG_2019_Mouse", "MSigDB_Hallmark_2020"]


def _weight_col(df):
    for c in ["avg_total_weight", "total_weight", "avg_weight", "weight",
              "score", "importance", "avg_ts_weight"]:
        if c in df.columns:
            return c
    return None


def load_tf_gene(path):
    df = pd.read_pickle(os.path.join(path, "pred_tf_gene.pkl"))
    df = df[~df["Gene"].astype(str).str.startswith("chr")].copy()
    df["Gene"] = df["Gene"].astype(str)
    return df


def gene_strength(df):
    wc = _weight_col(df)
    if wc is None:
        return df.groupby("Gene").size().astype(float)
    return df.groupby("Gene")[wc].max()


def celltype_expressed_background(adata_path, min_frac=0.05):
    """
    Fix 1: background = genes expressed in >= min_frac of cells of THIS cell type.
    Stricter (0.05) than before to remove ubiquitous housekeeping genes.
    """
    try:
        import anndata as ad
    except ImportError:
        print("  [bg] anndata missing -> genome-wide background")
        return None
    if not os.path.exists(adata_path):
        print(f"  [bg] {adata_path} missing -> genome-wide background")
        return None
    a = ad.read_h5ad(adata_path)
    n_expr = np.asarray((a.X > 0).sum(axis=0)).ravel()
    thresh = max(5, int(min_frac * a.n_obs))
    bg = set(map(str, a.var_names[n_expr >= thresh]))
    print(f"  [bg] cell-type expressed background: {len(bg)} genes (>= {thresh} cells)")
    return bg


def delta_specific(df_a, df_b, k=400, delta_min=0.05):
    """High-confidence A-specific genes (strong in A, weak/absent in B)."""
    sa, sb = gene_strength(df_a), gene_strength(df_b)
    allg = sa.index.union(sb.index)
    sa = sa.reindex(allg).fillna(0.0); sb = sb.reindex(allg).fillna(0.0)
    def _n(s):
        r = s.max() - s.min()
        return (s - s.min()) / r if r > 1e-9 else s * 0
    delta = _n(sa) - _n(sb)
    cand = delta[delta > delta_min].sort_values(ascending=False)
    return set(cand.head(k).index.astype(str))


def run_ora(gene_list, name, organism, libraries, background, out_csv=None):
    try:
        import gseapy as gp
    except ImportError:
        print(f"  [ORA] gseapy missing; skip {name}"); return None
    import time
    gene_list = [g for g in gene_list if isinstance(g, str) and not g.startswith("chr")]
    if len(gene_list) < 10:
        print(f"  [ORA] {name}: {len(gene_list)} genes (<10), skipped"); return None
    frames = []
    for gs in libraries:
        for attempt in range(3):
            kw = dict(gene_list=list(gene_list), gene_sets=gs, organism=organism,
                      outdir=None, no_plot=True)
            if background is not None:
                kw["background"] = list(background)
            try:
                enr = gp.enrichr(**kw); r = enr.results.copy()
                if r is not None and len(r) > 0:
                    r["library"] = gs; frames.append(r)
                break
            except Exception as e:
                if attempt < 2: time.sleep(2); continue
                print(f"  [ORA] {name}/{gs} failed: {e}")
    if not frames:
        print(f"  [ORA] {name}: no results"); return None
    res = pd.concat(frames, ignore_index=True)
    pcol = "Adjusted P-value" if "Adjusted P-value" in res.columns else "P-value"
    res["_padj"] = res[pcol]; res = res.sort_values("_padj").reset_index(drop=True)
    n_sig = int((res["_padj"] < SIG).sum())
    print(f"  [ORA] {name}: {len(res)} terms, {n_sig} sig "
          f"(top: {res.iloc[0]['Term'][:46]}, P_adj={res.iloc[0]['_padj']:.2g})")
    if out_csv: res.to_csv(out_csv, index=False)
    return res


def case2_focus(ad_base, out_dir):
    print("\n=== Case 2 AD — focused re-tuning (cell-type background) ===")
    os.makedirs(out_dir, exist_ok=True)
    M = {
        "CRND8_MG": os.path.join(ad_base, "model1_CRND8_Microglia/process/"),
        "WT_MG":    os.path.join(ad_base, "model2_WT_Microglia/process/"),
        "CRND8_AS": os.path.join(ad_base, "model3_CRND8_Astrocyte/process/"),
        "WT_AS":    os.path.join(ad_base, "model4_WT_Astrocyte/process/"),
    }
    nets = {k: load_tf_gene(v) for k, v in M.items()}

    for cell, crnd8, wt in [("microglia", "CRND8_MG", "WT_MG"),
                            ("astrocyte", "CRND8_AS", "WT_AS")]:
        print(f" -- {cell} --")
        bg = celltype_expressed_background(
            os.path.join(M[crnd8], "rna_processed.h5ad"), min_frac=0.05)
        spec = delta_specific(nets[crnd8], nets[wt])
        print(f"    CRND8-specific {cell}: {len(spec)} genes")
        run_ora(spec, f"focus_CRND8_{cell}", "mouse", LIBS_FOCUS_MOUSE, bg,
                os.path.join(out_dir, f"focus_GO_CRND8_{cell}.csv"))


if __name__ == "__main__":
    AD_BASE = "/home/wuyan/dygmamba_project/NewRealPlan/case2/data/AD/process/"
    OUT     = "/home/wuyan/dygmamba_project/DRIMA/data/case2/functional/"
    os.makedirs(OUT, exist_ok=True)
    case2_focus(AD_BASE, OUT)
    print("\nDone. Decision rule:")
    print("  - If microglia now shows neuroinflammation/lipid/complement and")
    print("    astrocyte shows reactive/complement/lipid -> real, write it up.")
    print("  - If microglia STILL shows ribosome/translation -> housekeeping")
    print("    contamination persists = AD signal is genuinely diffuse; report")
    print("    the diffuse-rewiring conclusion, do NOT force a pathway story.")
    print("  - Inspect *_shared_control equivalently; never report nominal-only")
    print("    hits as 'significant'.")


=== Case 2 AD — focused re-tuning (cell-type background) ===
 -- microglia --
  [bg] cell-type expressed background: 1995 genes (>= 33 cells)
    CRND8-specific microglia: 400 genes
  [ORA] focus_CRND8_microglia: 3529 terms, 9 sig (top: RNA splicing, via transesterification reaction, P_adj=0.0006)
 -- astrocyte --
  [bg] cell-type expressed background: 1995 genes (>= 39 cells)
    CRND8-specific astrocyte: 400 genes
  [ORA] focus_CRND8_astrocyte: 3426 terms, 0 sig (top: mRNA surveillance pathway, P_adj=0.076)

Done. Decision rule:
  - If microglia now shows neuroinflammation/lipid/complement and
    astrocyte shows reactive/complement/lipid -> real, write it up.
  - If microglia STILL shows ribosome/translation -> housekeeping
    contamination persists = AD signal is genuinely diffuse; report
    the diffuse-rewiring conclusion, do NOT force a pathway story.
  - Inspect *_shared_control equivalently; never report nominal-only
    hits as 'significant'.


## figure

In [23]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
DRIMA — Case 1 & Case 2 functional-enrichment FIGURE plotting (v3)
==================================================================
Fixes the REMAINING truncation: very long GO terms (e.g. "RNA splicing, via
transesterification reactions with bulged adenosine as nucleophile") were still
cut with '…' by the maxlen cap before wrapping. Now:
  - _clean_term maxlen raised to 100 (wrapping, not truncation, handles length)
  - TERM_ALIASES gives readable short names for the worst offenders
    (note this in the figure legend: "GO term names abbreviated for display")
  - wrap_width narrowed to 30 so multi-line labels fit the margin
  - non-significant bars stay grey (honest); Fig 4I annotations don't overlap title

Run AFTER drima_enrichment_v2.py and drima_AD_focus.py.
Deps: numpy pandas matplotlib
"""

import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 13, "axes.titlesize": 13, "axes.labelsize": 13,
    "xtick.labelsize": 11, "ytick.labelsize": 9, "figure.dpi": 100,
})
SIG = 0.05

# Readable display names for over-long GO/Reactome terms.
# (Identity unchanged; only the printed label is shortened. State this in the legend.)
TERM_ALIASES = {
    "RNA splicing, via transesterification reactions with bulged adenosine as nucleophile":
        "RNA splicing (spliceosomal)",
    "RNA splicing, via transesterification reactions":
        "RNA splicing (via transesterification)",
    "Processing Of Capped Intron-Containing Pre-mRNA":
        "Processing of capped intron-containing pre-mRNA",
    "regulation of phosphatidylinositol 3-kinase signaling":
        "regulation of PI3K signaling",
    "negative regulation of sphingolipid biosynthetic process":
        "neg. regulation of sphingolipid biosynthesis",
    "negative regulation of ceramide biosynthetic process":
        "neg. regulation of ceramide biosynthesis",
    "regulation of ceramide biosynthetic process":
        "regulation of ceramide biosynthesis",
    "regulation of hemoglobin biosynthetic process":
        "regulation of hemoglobin biosynthesis",
    "C-type lectin receptor signaling pathway":
        "C-type lectin receptor signaling",
}


# ---------------------------------------------------------------------------
def _read_ora(csv):
    if not os.path.exists(csv):
        print(f"  [warn] missing {csv}")
        return None
    df = pd.read_csv(csv)
    pcol = "Adjusted P-value" if "Adjusted P-value" in df.columns else "P-value"
    df["_padj"] = pd.to_numeric(df[pcol], errors="coerce")
    return df.sort_values("_padj").reset_index(drop=True)


def _clean_term(t, maxlen=100):
    """Strip GO/Reactome/WP id suffixes, apply alias, only truncate if absurdly long."""
    t = str(t)
    t = t.split(" (GO")[0]
    t = t.split(" R-HSA")[0].split(" R-MMU")[0].split(" WP")[0]
    t = t.strip()
    if t in TERM_ALIASES:
        return TERM_ALIASES[t]
    key = t.rstrip(", ")
    if key in TERM_ALIASES:
        return TERM_ALIASES[key]
    return t if len(t) <= maxlen else t[:maxlen - 1] + "…"


def _wrap(t, width=30):
    return "\n".join(textwrap.wrap(t, width=width)) if len(t) > width else t


def draw_ora_bar(ax, df, title, color, top_n=10, note=None, wrap_width=30):
    if df is None or len(df) == 0:
        ax.text(0.5, 0.5, f"{title}\n[no data]", ha="center", va="center")
        ax.axis("off"); return
    top = df.head(top_n).iloc[::-1]
    terms = [_wrap(_clean_term(t), wrap_width) for t in top["Term"]]
    vals = -np.log10(top["_padj"].clip(lower=1e-300))
    sig_mask = top["_padj"].values < SIG
    colors = [color if s else "#BBBBBB" for s in sig_mask]
    ax.barh(range(len(terms)), vals, color=colors, alpha=0.9, edgecolor="white")
    ax.set_yticks(range(len(terms)))
    ax.set_yticklabels(terms, fontsize=8.5)
    ax.set_xlabel(r"$-\log_{10}$ adjusted $P$")
    ax.set_title(title, fontsize=13, fontweight="bold", pad=14)
    ax.axvline(-np.log10(SIG), ls="--", c="grey", lw=1)
    ax.text(-np.log10(SIG), len(terms) - 0.3, " P=0.05", color="grey",
            fontsize=8, va="top")
    if note:
        ax.text(0.97, 0.04, note, transform=ax.transAxes, ha="right", va="bottom",
                fontsize=8.5, style="italic", color="#555555")
    ax.spines[["top", "right"]].set_visible(False)


def case2_main(c2_focus_dir, out_dir):
    print("=== Case 2 Fig 4H/4I (AD, honest diffuse) ===")
    os.makedirs(out_dir, exist_ok=True)
    mg = _read_ora(os.path.join(c2_focus_dir, "focus_GO_CRND8_microglia.csv"))
    asc = _read_ora(os.path.join(c2_focus_dir, "focus_GO_CRND8_astrocyte.csv"))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    draw_ora_bar(axes[0], mg, "Fig 4H  CRND8-specific microglia targets", "#2166AC",
                 note="broad RNA-processing,\nnot a disease-focal signature")
    draw_ora_bar(axes[1], asc, "Fig 4I  CRND8-specific astrocyte targets", "#762A83",
                 note="no term passes correction\n(diffuse rewiring)")
    if asc is not None and (asc["_padj"] < SIG).sum() == 0:
        axes[1].text(0.97, 0.13, "(no term passes P_adj<0.05)",
                     transform=axes[1].transAxes, ha="right", va="bottom",
                     fontsize=8.5, color="#762A83")
    fig.subplots_adjust(left=0.32, right=0.97, wspace=0.95, top=0.88, bottom=0.13)
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(out_dir, f"Fig4HI_AD_GO.{ext}"),
                    dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved Fig4HI_AD_GO.png/pdf -> {out_dir}")

In [24]:

AD_FOCUS = "/home/wuyan/dygmamba_project/DRIMA/data/case2/functional/"
OUT      = "/home/wuyan/dygmamba_project/DRIMA/data/case2/functional/"

case2_main(AD_FOCUS, OUT)
print("\nDone. No term is truncated now (wrapping + aliases).")
print("Add to the figure legend: 'Some GO term names abbreviated for display.'")
print("If a NEW long term appears, add it to TERM_ALIASES at the top.")

=== Case 2 Fig 4H/4I (AD, honest diffuse) ===
  saved Fig4HI_AD_GO.png/pdf -> /home/wuyan/dygmamba_project/DRIMA/data/case2/functional/

Done. No term is truncated now (wrapping + aliases).
Add to the figure legend: 'Some GO term names abbreviated for display.'
If a NEW long term appears, add it to TERM_ALIASES at the top.
